In [10]:
import jpype
import os
# jvm 강제 종료
if jpype.isJVMStarted():
	jpype.shutdownJVM()

# 환경 변수에 JAVA_HOME이라는 변수를 등록
# 일반 문자열에서 \는 기능이 있음. \문자를 하려면 \\해야함
os.environ['JAVA_HOME'] = r'C:\Users\C603\Documents\AWS Class\ide\jdk\jdk-21'

In [11]:
# 데이터 생성
import pandas as pd
dataset = pd.read_csv('sample.csv')
# dataset.head(5)
# len(dataset)

In [21]:
# 전처리 결측치가 있으면 삭제
dataset=dataset.dropna()

In [22]:
# pip install konlpy

In [34]:
from konlpy.tag import Okt
# 형태소 분석. 명사, 동사, 형용사를 추출

# 데이터 프레임에 있는 sentence열을 가져와서 리스트로 변환
sentence = dataset['sentence'].tolist()

# sentences => cleaner_sentences로 변환
# 원본 문자열 => 필요한 형태소만 추출한 문자열 

okt = Okt()
cleaned_sentence = []
# 형태소 단위로 쪼갬
for text in sentence:
	# stem=True : 동사를 동사 원형으로 바꿈
	result = okt.pos(text, stem=True)
	# 반복문 결과를 리스트를 저장
	words = [word for word, pos in result if pos in ['Noun', 'Verb', 'Adjective']]
	# for word, pos in result:
	# 	# print(f'{word}({pos})')
		
	# 	# 단어의 품사가 명사, 동사, 형용사이면 추출하여 리스트에 추가
	# 	if pos in ['Noun', 'Verb', 'Adjective']:
	# 		words.append(word)
	# 리스트에 있는 단어들을 문자열로 변환 => 명사, 동사, 형용사만 있는 문자으로 변환
	new_text = " ".join(words)
	cleaned_sentence.append(new_text)
# 결과물 id, sentence, label, cleaned_sentence
dataset['cleaned_sentence'] = cleaned_sentence
dataset



,id,sentence,label,cleaned_sentence
0,1,오늘 날씨가 정말 화창해서 기분이 최고예요,1,오늘 날씨 정말 화창하다 기분 최고
1,2,아침부터 일이 꼬여서 너무 짜증나요,0,아침 꼬이다 짜증나다
2,3,드디어 원하던 자격증 시험에 합격했습니다,1,원하다 자격증 시험 합격 하다
3,4,밤새 공부했는데 결과가 안 좋아서 우울해요,0,밤새 공부 하다 결과 안 좋다 우울하다
4,5,맛있는 음식을 먹으니 스트레스가 다 풀리네요,1,맛있다 음식 먹다 스트레스 풀리다
5,6,몸이 으슬으슬 떨리고 감기 기운이 있어요,0,몸 으슬으슬 떨리다 감기 기운 있다
6,7,친구들과 오랜만에 수다 떨어서 즐거웠어요,1,친구 만 수다 떨다 즐겁다
7,8,상사한테 혼나서 퇴사하고 싶은 마음뿐입니다,0,상사 혼나다 퇴사 싶다 마음 뿐이다
8,9,산책하며 좋아하는 노래를 들으니 행복해요,1,산책 하다 좋아하다 노래 들다 행복하다
9,10,길이 너무 막혀서 약속 시간에 늦을 것 같아요,0,길이 막히다 약속 시간 늦 것 같다


In [35]:
# 텍스트 전처리 함수
# 문장에서 명사, 동사, 형용사면 원형을 추출해서 문장을 만들어 반환
def text_preprocessing(text):
	# stem=True : 동사를 동사 원형으로 바꿈
	result = okt.pos(text, stem=True)
	# 반복문 결과를 리스트를 저장
	words = [word for word, pos in result if pos in ['Noun', 'Verb', 'Adjective']]
	return " ".join(words)

In [37]:
# text_preprocessing('오늘 날씨가 정말 화창해서 기분이 최고예요')
# '오늘 날씨 정말 화창하다 기분 최고'

dataset['cleaned_sentence2'] = dataset['sentence'].apply(text_preprocessing)
dataset[['cleaned_sentence','cleaned_sentence2']]

,cleaned_sentence,cleaned_sentence2
0,오늘 날씨 정말 화창하다 기분 최고,오늘 날씨 정말 화창하다 기분 최고
1,아침 꼬이다 짜증나다,아침 꼬이다 짜증나다
2,원하다 자격증 시험 합격 하다,원하다 자격증 시험 합격 하다
3,밤새 공부 하다 결과 안 좋다 우울하다,밤새 공부 하다 결과 안 좋다 우울하다
4,맛있다 음식 먹다 스트레스 풀리다,맛있다 음식 먹다 스트레스 풀리다
5,몸 으슬으슬 떨리다 감기 기운 있다,몸 으슬으슬 떨리다 감기 기운 있다
6,친구 만 수다 떨다 즐겁다,친구 만 수다 떨다 즐겁다
7,상사 혼나다 퇴사 싶다 마음 뿐이다,상사 혼나다 퇴사 싶다 마음 뿐이다
8,산책 하다 좋아하다 노래 들다 행복하다,산책 하다 좋아하다 노래 들다 행복하다
9,길이 막히다 약속 시간 늦 것 같다,길이 막히다 약속 시간 늦 것 같다


In [42]:
# 벡터화
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer()
# 독립, 종속 분리
x = vectorizer.fit_transform(dataset['cleaned_sentence'])
y = dataset['label']

from sklearn.model_selection import train_test_split
# 원래는 훈련, 테스트 분리하여 학습 후의 정확도를 확인
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2)

In [43]:
# 학습
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(x_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [44]:
model.score(x_test, y_test)

0.625

In [50]:
# 테스트
text = '자격증 시험에 합격했습니다'
vectir_text = vectorizer.transform([text])
res = model.predict(vectir_text)
print(f'{text} : {'긍정' if res[0] else '부정'}')

# 시험 떨어지다라는 데이터를 학습하지 못해서 결과가 정확하지 않음
text = '날씨가 화창한데 자격증 시험에 떨어졌습니다'
vectir_text = vectorizer.transform([text])
res = model.predict(vectir_text)
print(f'{text} : {'긍정' if res[0] else '부정'}')

자격증 시험에 합격했습니다 : 긍정
날씨가 화창한데 자격증 시험에 떨어졌습니다 : 긍정
